<h1>07 - Advanced Text Generation Techniques and Tools</h1>
<i>Going beyond prompt engineering.</i>


In [ ]:
# %%capture
# !pip install langchain>=0.1.17 openai>=1.13.3 langchain_openai>=0.1.6 transformers>=4.40.1 datasets>=2.18.0 accelerate>=0.27.2 sentence-transformers>=2.5.1 duckduckgo-search>=5.2.2 langchain_community
# !CMAKE_ARGS="-DLLAMA_CUDA=on" pip install llama-cpp-python==0.2.69

# Loading an LLM

In [ ]:
# !wget https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf

# If this command does not work for you, you can use the link directly to download the model
# https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf

--2026-09-17 22:26:45--  https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf
Resolving huggingface.co (huggingface.co)... 52.222.136.38, 52.222.136.92, 52.222.136.117, ...
Connecting to huggingface.co (huggingface.co)|52.222.136.38|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.aws.cdn.hf.co/xet-bridge-us/662698108f7573e6a6478546/a9cdcf6e9514941ea9e596583b3d3c44dd99359fb7dd57f322bb84a0adc12ad4?X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27Phi-3-mini-4k-instruct-fp16.gguf%3B+filename%3D%22Phi-3-mini-4k-instruct-fp16.gguf%22%3B&user_id=public&Expires=1789675006&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5hd3MuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjYyNjk4MTA4Zjc1NzNlNmE2NDc4NTQ2L2E5Y2RjZjZlOTUxNDk0MWVhOWU1OTY1ODNiM2QzYzQ0ZGQ5OTM1OWZiN2RkNTdmMzIyYmI4NGEwYWRjMTJhZDRcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZyZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPWlubGluZSUzQitm

OSError: [Errno 5] Input/output error

In [7]:
from langchain import LlamaCpp


In [8]:
absolute_path='Phi-3-mini-4k-instruct-fp16.gguf'

In [9]:
from langchain import LlamaCpp

# Make sure the model path is correct for your system!
llm = LlamaCpp(
    model_path=absolute_path,
    n_gpu_layers=-1,
    max_tokens=500,
    n_ctx=2048,
    seed=42,
    verbose=False
)

In [5]:
LlamaCpp??

Init signature:
LlamaCpp(
    *args: Any,
    name: Optional[str] = None,
    cache: ForwardRef('Union[BaseCache, bool, None]') = None,
    verbose: bool = True,
    callbacks: ForwardRef('Callbacks') = None,
    tags: Optional[List[str]] = None,
    metadata: Optional[Dict[str, Any]] = None,
    custom_get_token_ids: Optional[Callable[[str], List[int]]] = None,
    callback_manager: Optional[langchain_core.callbacks.base.BaseCallbackManager] = None,
    client: Any = None,
    model_path: str,
    lora_base: Optional[str] = None,
    lora_path: Optional[str] = None,
    n_ctx: int = 512,
    n_parts: int = -1,
    seed: int = -1,
    f16_kv: bool = True,
    logits_all: bool = False,
    vocab_only: bool = False,
    use_mlock: bool = False,
    n_threads: Optional[int] = None,
    n_batch: Optional[int] = 8,
    n_gpu_layers: Optional[int] = None,
    suffix: Optional[str] = None,
    max_tokens: Optional[int] = 256,
    temperature: Optional[float] = 0.8,
    top_p: Optional[float] 

In [10]:
llm.invoke("Hi! My name is Maarten. What is 1 + 1?")

''

### Chains

In [11]:
from langchain import PromptTemplate

# Create a prompt template with the "input_prompt" variable
template = """<s><|user|>
{input_prompt}<|end|>
<|assistant|>"""
prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt"]
)

In [12]:
basic_chain = prompt | llm

In [13]:
basic_chain

PromptTemplate(input_variables=['input_prompt'], template='<s><|user|>\n{input_prompt}<|end|>\n<|assistant|>')
| LlamaCpp(verbose=False, client=<llama_cpp.llama.Llama object at 0x10d2d4e10>, model_path='Phi-3-mini-4k-instruct-fp16.gguf', n_ctx=2048, seed=42, n_gpu_layers=-1, max_tokens=500)

In [ ]:
# basic_chain.invoke({'Hi my name is Alex. tell me how old are you and give me the answer for radical two '})

/opt/anaconda3/envs/thellmbook/lib/python3.11/site-packages/llama_cpp/llama.py:1031: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


" Hello Alex! I'm an AI, so I don't have an age in the traditional sense. However, my technology has been developed over many years by a team of engineers and researchers. Regarding your question about the square root of 2 (radical 2), it is approximately equal to 1.41421356237. This number is an irrational number, which means it cannot be exactly expressed as a simple fraction and its decimal representation goes on forever without repeating."

In [15]:
# Use the chain
basic_chain.invoke(
    {
        "input_prompt": "Hi! My name is Maarten. What is 1 + 1?",
    }
)

/opt/anaconda3/envs/thellmbook/lib/python3.11/site-packages/llama_cpp/llama.py:1031: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


' Hello Maarten! The answer to 1 + 1 is 2.'

### Multiple Chains

In [17]:
from langchain import LLMChain

# Create a chain for the title of our story
template = """<s><|user|>
Create a title for a story about {summary}. Only return the title.<|end|>
<|assistant|>"""
title_prompt = PromptTemplate(template=template, input_variables=["summary"])
title = LLMChain(llm=llm, prompt=title_prompt, output_key="title")

/var/folders/_n/5z_drbxd3c3dbd1xkw7qsyyh0000gn/T/ipykernel_11524/3105618174.py:8: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 0.3.0. Use RunnableSequence, e.g., `prompt | llm` instead.
  title = LLMChain(llm=llm, prompt=title_prompt, output_key="title")


In [18]:
title.invoke({"summary": "a girl that lost her mother"})

/opt/anaconda3/envs/thellmbook/lib/python3.11/site-packages/llama_cpp/llama.py:1031: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'summary': 'a girl that lost her mother',
 'title': ' "Whispers of Loss: A Tale of a Mother\'s Absence"'}

In [20]:
# Create a chain for the character description using the summary and title
template = """<s><|user|>
Describe the main character of a story about {summary} with the title {title}. Use only two sentences.<|end|>
<|assistant|>"""
character_prompt = PromptTemplate(
    template=template, input_variables=["summary", "title"]
)
character = LLMChain(llm=llm, prompt=character_prompt, output_key="character")

In [21]:
# Create a chain for the story using the summary, title, and character description
template = """<s><|user|>
Create a story about {summary} with the title {title}. The main charachter is: {character}. Only return the story and it cannot be longer than one paragraph<|end|>
<|assistant|>"""
story_prompt = PromptTemplate(
    template=template, input_variables=["summary", "title", "character"]
)
story = LLMChain(llm=llm, prompt=story_prompt, output_key="story")

In [22]:
# Combine all three components to create the full chain
llm_chain = title | character | story

In [23]:
llm_chain.invoke("a girl that lost her mother")

/opt/anaconda3/envs/thellmbook/lib/python3.11/site-packages/llama_cpp/llama.py:1031: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
/opt/anaconda3/envs/thellmbook/lib/python3.11/site-packages/llama_cpp/llama.py:1031: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
/opt/anaconda3/envs/thellmbook/lib/python3.11/site-packages/llama_cpp/llama.py:1031: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'summary': 'a girl that lost her mother',
 'title': ' "Echoes of Love: A Journey Through Loss and Healing"',
 'character': " The protagonist, Maya, is an introspective teenager who has been grappling with unconditional love and profound loss after losing her mother in a tragic accident at an early age. Her strength, resilience, and journey towards healing are beautifully illustrated as she navigates the complex emotions of grief while carrying on her mother's legacy through selfless acts of kindness and compassion for others.",
 'story': ' "Echoes of Love: A Journey Through Loss and Healing" follows Maya, an introspective teenager whose world was shattered when her beloved mother passed away in a tragic accident. Despite the unbearable weight of grief that threatened to consume her tender heart, Maya found solace in embracing the boundless love her mother had showered upon her as she navigated through life\'s ups and downs. Through each act of kindness – from volunteering at a local s

# Memory

In [24]:
# Let's give the LLM our name
basic_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})

/opt/anaconda3/envs/thellmbook/lib/python3.11/site-packages/llama_cpp/llama.py:1031: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


" Hello Maarten! The answer to 1 + 1 is 2. It's a basic arithmetic operation where you add one unit to another, resulting in two units altogether."

In [25]:
# Next, we ask the LLM to reproduce the name
basic_chain.invoke({"input_prompt": "What is my name?"})

/opt/anaconda3/envs/thellmbook/lib/python3.11/site-packages/llama_cpp/llama.py:1031: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


" I'm unable to determine your name as I don't have the ability to access personal data or information about individuals. If you need assistance with something else, feel free to ask!"

## ConversationBuffer

In [30]:
# Create an updated prompt template to include a chat history
template = """<s><|user|>Current conversation:{chat_history}

{input_prompt}<|end|>
<|assistant|>"""

prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt", "chat_history"]
)

In [31]:
from langchain.memory import ConversationBufferMemory

# Define the type of Memory we will use
memory = ConversationBufferMemory(memory_key="chat_history")

# Chain the LLM, Prompt, and Memory together
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

In [32]:
# Generate a conversation and ask a basic question
llm_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})

/opt/anaconda3/envs/thellmbook/lib/python3.11/site-packages/llama_cpp/llama.py:1031: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'input_prompt': 'Hi! My name is Maarten. What is 1 + 1?',
 'chat_history': '',
 'text': " The answer to 1 + 1 is 2. It's a basic arithmetic operation where you add one unit to another, resulting in two units altogether. If you have any further questions or need assistance with something else, feel free to ask!"}

In [33]:
# Does the LLM remember the name we gave it?
llm_chain.invoke({"input_prompt": "What is my name?"})

/opt/anaconda3/envs/thellmbook/lib/python3.11/site-packages/llama_cpp/llama.py:1031: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'input_prompt': 'What is my name?',
 'chat_history': "Human: Hi! My name is Maarten. What is 1 + 1?\nAI:  The answer to 1 + 1 is 2. It's a basic arithmetic operation where you add one unit to another, resulting in two units altogether. If you have any further questions or need assistance with something else, feel free to ask!",
 'text': ' Your name is Maarten.\n\nI\'m an AI and do not have the ability to know personal details unless shared during our conversation. In this case, you mentioned your name in the beginning, so I understand it as "Maarten." However, please remember that I don\'t have access to personal data unless it is provided within our interaction for the purpose of assisting you.'}

## ConversationBufferMemoryWindow

In [34]:
from langchain.memory import ConversationBufferWindowMemory

# Retain only the last 2 conversations in memory
memory = ConversationBufferWindowMemory(k=2, memory_key="chat_history")

# Chain the LLM, Prompt, and Memory together
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

In [35]:
# Ask two questions and generate two conversations in its memory
llm_chain.invoke({"input_prompt":"Hi! My name is Maarten and I am 33 years old. What is 1 + 1?"})
llm_chain.invoke({"input_prompt":"What is 3 + 3?"})

/opt/anaconda3/envs/thellmbook/lib/python3.11/site-packages/llama_cpp/llama.py:1031: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
/opt/anaconda3/envs/thellmbook/lib/python3.11/site-packages/llama_cpp/llama.py:1031: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'input_prompt': 'What is 3 + 3?',
 'chat_history': 'Human: Hi! My name is Maarten and I am 33 years old. What is 1 + 1?\nAI:  The answer to the mathematical question you asked, "What is 1 + 1?" is 2. Despite sharing a personal introduction in your message, the core of our conversation remains focused on the arithmetic operation requested.\n\nRegarding privacy and data protection, I don\'t retain personal information shared during our interactions. So, while we discuss various topics, any details you provide are not stored or used beyond this immediate exchange.',
 'text': ' The answer to your new arithmetic question "What is 3 + 3?" is 6. Again, as an AI, I\'m here to assist with the information and tasks at hand without storing personal data shared in our conversation.\n\nIf you have any mathematical questions or need assistance with other topics within my capabilities, feel free to ask!'}

In [36]:
# Check whether it knows the name we gave it
llm_chain.invoke({"input_prompt":"What is my name?"})

/opt/anaconda3/envs/thellmbook/lib/python3.11/site-packages/llama_cpp/llama.py:1031: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'input_prompt': 'What is my name?',
 'chat_history': 'Human: Hi! My name is Maarten and I am 33 years old. What is 1 + 1?\nAI:  The answer to the mathematical question you asked, "What is 1 + 1?" is 2. Despite sharing a personal introduction in your message, the core of our conversation remains focused on the arithmetic operation requested.\n\nRegarding privacy and data protection, I don\'t retain personal information shared during our interactions. So, while we discuss various topics, any details you provide are not stored or used beyond this immediate exchange.\nHuman: What is 3 + 3?\nAI:  The answer to your new arithmetic question "What is 3 + 3?" is 6. Again, as an AI, I\'m here to assist with the information and tasks at hand without storing personal data shared in our conversation.\n\nIf you have any mathematical questions or need assistance with other topics within my capabilities, feel free to ask!',
 'text': " Your name, based on your initial message, appears to be Maarten. H

In [37]:
# Check whether it knows the age we gave it
llm_chain.invoke({"input_prompt":"What is my age?"})

/opt/anaconda3/envs/thellmbook/lib/python3.11/site-packages/llama_cpp/llama.py:1031: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'input_prompt': 'What is my age?',
 'chat_history': 'Human: What is 3 + 3?\nAI:  The answer to your new arithmetic question "What is 3 + 3?" is 6. Again, as an AI, I\'m here to assist with the information and tasks at hand without storing personal data shared in our conversation.\n\nIf you have any mathematical questions or need assistance with other topics within my capabilities, feel free to ask!\nHuman: What is my name?\nAI:  Your name, based on your initial message, appears to be Maarten. However, it\'s important to note that as an AI, I don\'t store personal data from our conversations for privacy reasons. Each interaction with me doesn\'t retain past information unless specifically programmed to do so for the purpose of improving user experience in future sessions within the confines of data protection regulations. So, while you mentioned your name earlier, it won\'t be remembered after this conversation ends.',
 'text': " I'm unable to determine your age as I don't have the cap

## ConversationSummary

In [39]:
# Create a summary prompt template
summary_prompt_template = """<s><|user|>Summarize the conversations and update with the new lines.

Current summary:
{summary}

new lines of conversation:
{new_lines}

New summary:<|end|>
<|assistant|>"""
summary_prompt = PromptTemplate(
    input_variables=["new_lines", "summary"],
    template=summary_prompt_template
)

In [42]:
from langchain.memory import ConversationSummaryMemory

# Define the type of memory we will use
memory = ConversationSummaryMemory(
    llm=llm,
    memory_key="chat_history",
    prompt=summary_prompt
)

# Chain the LLM, prompt, and memory together
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

In [43]:
# Generate a conversation and ask for the name
llm_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})
llm_chain.invoke({"input_prompt": "What is my name?"})

/opt/anaconda3/envs/thellmbook/lib/python3.11/site-packages/llama_cpp/llama.py:1031: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
/opt/anaconda3/envs/thellmbook/lib/python3.11/site-packages/llama_cpp/llama.py:1031: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
/opt/anaconda3/envs/thellmbook/lib/python3.11/site-packages/llama_cpp/llama.py:1031: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
/opt/anaconda3/envs/thellmbook/lib/python3.11/site-packages/llama_cpp/llama.py:1031: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'input_prompt': 'What is my name?',
 'chat_history': ' Human introduces themselves as Maarten and asks the AI for the result of 1+1, which the AI correctly responds with the answer being 2, explaining it as a fundamental addition operation yielding two units.',
 'text': ' Your name mentioned in this context is Maarten. How can I assist you further?'}

In [44]:
# Check whether it has summarized everything thus far
llm_chain.invoke({"input_prompt": "What was the first question I asked?"})

/opt/anaconda3/envs/thellmbook/lib/python3.11/site-packages/llama_cpp/llama.py:1031: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
/opt/anaconda3/envs/thellmbook/lib/python3.11/site-packages/llama_cpp/llama.py:1031: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'input_prompt': 'What was the first question I asked?',
 'chat_history': ' Maarten introduces themselves and inquires about the result of 1+1, to which the AI responds with 2 as a basic addition operation combining two units. When asked about their name, the AI confirms it is Maarten and offers additional assistance.',
 'text': ' The first question you asked was "to which the AI responds with 2 as a basic addition operation combining two units." However, if we consider the sequence of actions in your scenario, the very first thing you did was introduce yourself and Maarten inquiring about the result of 1+1. So technically, the first question implied through conversation is "what is the result of 1+1?"'}

In [ ]:
# Check what the summary is thus far
memory.load_memory_variables({})

{'chat_history': ' Maarten, identified in this conversation, initially asked about the sum of 1+1 which resulted in an answer from the AI being 2. Subsequently, he sought clarification on his name but the AI informed him that no personal data is retained beyond a single session due to privacy reasons. The AI then offered further assistance if required. Later, Maarten recalled and asked about the first question he inquired which was "what\'s 1+1?"'}

# Agents

In [45]:
import os
from langchain_openai import ChatOpenAI

# Load OpenAI's LLMs with LangChain
os.environ["OPENAI_API_KEY"] = "MY_KEY"
openai_llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

In [46]:
# Create the ReAct template
react_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate(
    template=react_template,
    input_variables=["tools", "tool_names", "input", "agent_scratchpad"]
)

In [47]:
from langchain.agents import load_tools, Tool
from langchain.tools import DuckDuckGoSearchResults

# You can create the tool to pass to an agent
search = DuckDuckGoSearchResults()
search_tool = Tool(
    name="duckduck",
    description="A web search engine. Use this to as a search engine for general queries.",
    func=search.run,
)



In [48]:
# Prepare tools
tools = load_tools(["llm-math"], llm=openai_llm)
tools.append(search_tool)

In [49]:
print(tools)

[Tool(name='Calculator', description='Useful for when you need to answer questions about math.', func=<bound method Chain.run of LLMMathChain(llm_chain=LLMChain(prompt=PromptTemplate(input_variables=['question'], template='Translate a math problem into a expression that can be executed using Python\'s numexpr library. Use the output of running this code to answer the question.\n\nQuestion: ${{Question with math problem.}}\n```text\n${{single line mathematical expression that solves the problem}}\n```\n...numexpr.evaluate(text)...\n```output\n${{Output of running the code}}\n```\nAnswer: ${{Answer}}\n\nBegin.\n\nQuestion: What is 37593 * 67?\n```text\n37593 * 67\n```\n...numexpr.evaluate("37593 * 67")...\n```output\n2518731\n```\nAnswer: 2518731\n\nQuestion: 37593^(1/5)\n```text\n37593**(1/5)\n```\n...numexpr.evaluate("37593**(1/5)")...\n```output\n8.222831614237718\n```\nAnswer: 8.222831614237718\n\nQuestion: {question}\n'), llm=ChatOpenAI(client=<openai.resources.chat.completions.Comp

In [50]:
from langchain.agents import AgentExecutor, create_react_agent

# Construct the ReAct agent
agent = create_react_agent(openai_llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent, tools=tools, verbose=True, handle_parsing_errors=True
)

In [51]:
# What is the Price of a MacBook Pro?
agent_executor.invoke(
    {
        "input": "What is the current price of a MacBook Pro in USD? How much would it cost in EUR if the exchange rate is 0.85 EUR for 1 USD?"
    }
)



> Entering new AgentExecutor chain...


APIConnectionError: Connection error.